## 1. 실행 환경 및 패키지

T4 GPU 런타임을 선택하세요. Colab의 CUDA PyTorch를 사용합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys, json, shutil

root = Path('/content/noonchi_complete')
root.mkdir(exist_ok=True)
for name, source in {'common.py': 'import unicodedata\n\nLABEL_TO_ID = {"Non-Sarcasm": 0, "Sarcasm": 1}\nSPLITS = ("train", "validation", "test")\n\n\ndef normalize(value):\n    return " ".join(unicodedata.normalize("NFC", value).split())\n\n\ndef format_input(context, response):\n    return f"맥락: {normalize(context)}\\n마지막 발언: {normalize(response)}"\n', 'preprocess.py': 'import hashlib\nimport itertools\nimport json\nfrom pathlib import Path\nimport re\nimport unicodedata\nimport sys\n\nimport pandas as pd\n\nroot = Path.cwd()\nsource = Path(sys.argv[1]) if len(sys.argv) > 1 else root.parent\nfields = ["context", "response", "label", "sarcasm_explanation"]\nlabels = ["Sarcasm", "Non-Sarcasm"]\nframes = []\nreport = {"files": {}, "splits": {}, "overlap": {}}\n\n\ndef normalize(value):\n    return " ".join(unicodedata.normalize("NFC", value).split()) if isinstance(value, str) else ""\n\n\ndef distribution(series):\n    return {key: round(float(value), 2) for key, value in {\n        "min": series.min(), "mean": series.mean(), "p50": series.median(),\n        "p90": series.quantile(.90), "p95": series.quantile(.95),\n        "p99": series.quantile(.99), "max": series.max()\n    }.items()}\n\n\ndef lengths(frame):\n    context, response = frame["ctx"], frame["resp"]\n    combined = (context + " " + response).str.strip()\n    return {\n        "context_chars": distribution(context.str.len()),\n        "response_chars": distribution(response.str.len()),\n        "combined_chars": distribution(combined.str.len()),\n        "whitespace_tokens": distribution(combined.str.split().str.len()),\n        "speaker_turns": distribution(combined.map(lambda s: len(re.findall(r"(?<!\\S)[AB]\\s*:", s)))),\n    }\n\n\nfor split in ("train", "validation", "test"):\n    path = source / f"{split}-00000-of-00001.parquet"\n    frame = pd.read_parquet(path, engine="pyarrow")\n    report["files"][split] = {\n        "file": path.name, "bytes": path.stat().st_size,\n        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "read_success": True,\n    }\n    missing = {f: {\n        "null": int(frame[f].isna().sum()), "empty": int(frame[f].eq("").sum()),\n        "whitespace_only": int(frame[f].map(lambda x: isinstance(x, str) and bool(x) and not x.strip()).sum())\n    } for f in fields}\n    exact = int(frame.duplicated().sum())\n    frame = frame.assign(ctx=frame.context.map(normalize), resp=frame.response.map(normalize),\n                         lab=frame.label.map(normalize), split=split, row=range(len(frame)))\n    frame["valid"] = frame.ctx.ne("") & frame.resp.ne("") & frame.lab.isin(labels)\n    valid = frame[frame.valid]\n    keys = ["ctx", "resp"]\n    counts = valid.groupby(keys).lab.agg(["size", "nunique"])\n    report["splits"][split] = {\n        "rows": len(frame), "labels": frame.lab.value_counts().to_dict(), "missing": missing,\n        "exact_extra": exact, "normalized_extra_valid": int((counts["size"] - 1).sum()),\n        "normalized_duplicate_groups_valid": int(counts["size"].gt(1).sum()),\n        "conflict_groups_valid": int(counts["nunique"].gt(1).sum()),\n        "invalid_required_rows": int((~frame.valid).sum()),\n        "normalized_changed_fields": {a: int((frame[a].notna() & frame[a].ne(frame[b])).sum())\n                                       for a, b in [("context", "ctx"), ("response", "resp"), ("label", "lab")]},\n        "explanation_blank_by_label": frame.assign(blank=frame.sarcasm_explanation.map(normalize).eq(""))\n            .groupby("lab").blank.sum().to_dict(),\n        "lengths_valid": lengths(valid),\n    }\n    frames.append(frame)\n\nall_rows = pd.concat(frames, ignore_index=True)\nvalid = all_rows[all_rows.valid].copy()\ncounts = valid.groupby(keys).lab.agg(["size", "nunique"])\nreport["totals"] = {\n    "rows": len(all_rows), "bytes": sum(f["bytes"] for f in report["files"].values()),\n    "labels": all_rows.lab.value_counts().to_dict(),\n    "exact_extra": int(all_rows[fields].duplicated().sum()),\n    "normalized_extra_valid": int((counts["size"] - 1).sum()),\n    "conflict_groups_valid": int(counts["nunique"].gt(1).sum()),\n    "lengths_valid": lengths(valid),\n}\nfor a, b in itertools.combinations(frames, 2):\n    av, bv = a[a.valid], b[b.valid]\n    pair = {}\n    for name, columns in [("exact_rows", fields), ("input", keys), ("context", ["ctx"]), ("response", ["resp"])]:\n        left, right = (a, b) if name == "exact_rows" else (av, bv)\n        if name == "exact_rows":\n            left, right = left.fillna("<NULL>"), right.fillna("<NULL>")\n        ka = pd.MultiIndex.from_frame(left[columns])\n        kb = pd.MultiIndex.from_frame(right[columns])\n        common = ka.unique().intersection(kb.unique())\n        pair[name] = {"keys": len(common), "left_rows": int(ka.isin(common).sum()),\n                      "right_rows": int(kb.isin(common).sum())}\n    def neutralize_speakers(s):\n        return re.sub(r"(?<!\\S)[AB]\\s*:", "S:", s)\n\n    for name, columns in [("speaker_neutral_input", keys), ("speaker_neutral_context", ["ctx"])]:\n        ka = pd.MultiIndex.from_frame(av[columns].apply(lambda s: s.map(neutralize_speakers)))\n        kb = pd.MultiIndex.from_frame(bv[columns].apply(lambda s: s.map(neutralize_speakers)))\n        common = ka.unique().intersection(kb.unique())\n        pair[name] = {"keys": len(common), "left_rows": int(ka.isin(common).sum()),\n                      "right_rows": int(kb.isin(common).sum())}\n    report["overlap"][a.split.iloc[0] + "/" + b.split.iloc[0]] = pair\n\nconflicts = counts[counts["nunique"].gt(1)].index\nvalid["conflict"] = pd.MultiIndex.from_frame(valid[keys]).isin(conflicts)\n# Preserve held-out examples ahead of development and training rows.\npriority = {"test": 0, "validation": 1, "train": 2}\nclean = valid[~valid.conflict].copy()\nclean = clean.assign(priority=clean.split.map(priority)).sort_values(["priority", "row"])\nclean = clean.drop_duplicates(keys)\nreport["preprocessed"] = clean.split.value_counts().to_dict()\nreport["kept_rows"] = len(clean)\nreport["removed_rows"] = len(all_rows) - len(clean)\nreport["removed_breakdown"] = {\n    "invalid_required": int((~all_rows.valid).sum()),\n    "conflicting_labels": int(valid.conflict.sum()),\n    "duplicate_input": int(len(valid) - valid.conflict.sum() - len(clean)),\n}\nout = Path(sys.argv[2]) if len(sys.argv) > 2 else root / "outputs"\nout.mkdir(exist_ok=True)\nclean[["split", "row", "ctx", "resp", "lab"]].rename(\n    columns={"row": "source_row_zero_based", "ctx": "context", "resp": "response", "lab": "label"}\n).to_parquet(out / "kocosa_preprocessed.parquet", index=False)\nreport["provenance"] = "All three input Parquet files were read; see files for SHA-256 and byte counts."\nreport["tokenization"] = "Whitespace token counts are measured. KoELECTRA subword counts are unavailable without its vocabulary/tokenizer."\nreport["redistribution"] = "Local analysis and preprocessing only. Dataset license and redistribution permission remain unverified."\n(out / "kocosa_audit.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\nprint(json.dumps(report, ensure_ascii=True))\n', 'train.py': 'import json\nfrom pathlib import Path\nimport sys\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score\nfrom sklearn.pipeline import make_pipeline\n\nfrom common import LABEL_TO_ID, SPLITS, format_input\n\n\ndef metrics(y, pred):\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return dict(accuracy=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average="macro"),\n                precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),\n                fpr=float(fp / (tn + fp)), confusion=[[int(tn), int(fp)], [int(fn), int(tp)]])\n\n\ndef run(source, audit_file=None, output="results"):\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=True)\n    frame = pd.read_parquet(source)\n    frame["target"] = frame.label.map(LABEL_TO_ID)\n    assert frame.target.notna().all() and not frame.duplicated(["context", "response"]).any()\n    train, val, test = [frame[frame.split.eq(s)].reset_index(drop=True) for s in SPLITS]\n    results = {}\n    for mode in ("context", "response_only"):\n        def features(rows):\n            return [format_input(c if mode == "context" else "", r) for c, r in zip(rows.context, rows.response)]\n\n        x_train, x_val, x_test = map(features, (train, val, test))\n        trials = []\n        best = None\n        for c in (.25, 1, 4):\n            for weight in (None, "balanced"):\n                model = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2,\n                                                       max_features=85000, sublinear_tf=True),\n                                      LogisticRegression(C=c, class_weight=weight, max_iter=500, random_state=42))\n                model.fit(x_train, train.target)\n                score = f1_score(val.target, model.predict(x_val), average="macro")\n                trials.append(dict(C=c, class_weight=weight, validation_macro_f1=score))\n                if best is None or score > best[0]:\n                    best = (score, model, trials[-1])\n        _, model, selection = best\n        pred = model.predict(x_test)\n        results[mode] = dict(selection=selection, trials=trials, test=metrics(test.target, pred))\n        errors = test.loc[test.target.ne(pred), ["source_row_zero_based", "label"]].copy()\n        errors["prediction"] = pred[test.target.ne(pred)]\n        errors["score"] = model.predict_proba(x_test)[test.target.ne(pred), 1]\n        errors.to_csv(out / f"errors_{mode}.csv", index=False)\n        fig, ax = plt.subplots()\n        ConfusionMatrixDisplay(confusion_matrix(test.target, pred), display_labels=["Non-Sarcasm", "Sarcasm"]).plot(ax=ax)\n        fig.tight_layout()\n        fig.savefig(out / f"confusion_{mode}.png", dpi=135)\n        plt.close(fig)\n        if mode == "context":\n            joblib.dump(model, out / "baseline.joblib", compress=3)\n    fig, ax = plt.subplots()\n    for mode, result in results.items():\n        for weight in (None, "balanced"):\n            trials = [x for x in result["trials"] if x["class_weight"] == weight]\n            ax.plot([x["C"] for x in trials], [x["validation_macro_f1"] for x in trials],\n                    "o-", label=f"{mode} / {weight or \'unweighted\'}")\n    ax.set(xscale="log", xlabel="Logistic regression C", ylabel="Validation Macro F1")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / "validation.png", dpi=135)\n    plt.close(fig)\n    if audit_file:\n        audit = json.loads(Path(audit_file).read_text(encoding="utf-8"))\n        after = pd.crosstab(frame.split, frame.label)\n        before = pd.DataFrame({s: data["labels"] for s, data in audit["splits"].items()}).T\n        comparison = pd.concat({"before": before, "after": after}, names=["stage", "split"])\n        comparison.to_csv(out / "preprocessing_counts.csv")\n        ax = comparison.plot.bar(rot=30)\n        ax.set_ylabel("Rows")\n        ax.figure.tight_layout()\n        ax.figure.savefig(out / "preprocessing.png", dpi=135)\n        plt.close(ax.figure)\n    (out / "metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")\n    print(json.dumps(results, indent=2))\n\n\nif __name__ == "__main__":\n    run(*sys.argv[1:])\n', 'finetune.py': 'import argparse\nimport json\nimport hashlib\nimport math\nfrom pathlib import Path\nimport tempfile\nfrom importlib.metadata import version\n\nfrom datasets import Dataset\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport torch\nfrom packaging.version import Version\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)\n\nfrom train import metrics\nfrom common import LABEL_TO_ID, SPLITS\n\n\ndef load_splits(path):\n    frame = pd.read_parquet(path)\n    frame["labels"] = frame.label.map(LABEL_TO_ID)\n    for column in ("context", "response"):\n        if not frame[column].map(lambda value: isinstance(value, str) and bool(value.strip())).all():\n            raise ValueError(f"Missing or blank {column} in preprocessed data")\n    if frame.labels.isna().any() or frame.duplicated(["context", "response"]).any():\n        raise ValueError("Unknown labels or duplicate inputs in preprocessed data")\n    if set(frame.split.unique()) != set(SPLITS):\n        raise ValueError("Expected nonempty train, validation and test splits")\n    if any(group.labels.nunique() != 2 for _, group in frame.groupby("split")):\n        raise ValueError("Each split must contain both classes")\n    return {split: frame[frame.split.eq(split)].reset_index(drop=True) for split in SPLITS}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("data", type=Path)\n    parser.add_argument("--output", type=Path, default=Path("koelectra_results"))\n    parser.add_argument("--batch-size", type=int, choices=(2, 4, 8), default=8)\n    args = parser.parse_args()\n    if Version(torch.__version__.split("+", 1)[0]) < Version("2.6"):\n        raise RuntimeError("Use a Colab GPU runtime with PyTorch 2.6 or newer to load the KoELECTRA checkpoint.")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select a GPU runtime in Colab before running this experiment.")\n    subsets = load_splits(args.data)\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(42)\n    with tempfile.TemporaryDirectory(prefix="noonchi-checkpoints-") as scratch:\n        train_and_evaluate(args, subsets, scratch)\n\n\ndef train_and_evaluate(args, subsets, scratch):\n    checkpoint = "monologg/koelectra-base-v3-discriminator"\n    tokenizer = AutoTokenizer.from_pretrained(checkpoint)\n    distributions, lengths = {}, {}\n    for split, subset in subsets.items():\n        ids = tokenizer(list(subset.context), list(subset.response), truncation=False, verbose=False)["input_ids"]\n        lengths[split] = pd.Series([len(x) for x in ids])\n    max_length = min(tokenizer.model_max_length, 512,\n                     max(64, 8 * math.ceil(lengths["train"].quantile(.95) / 8)))\n    for split, values in lengths.items():\n        distributions[split] = dict(summary=values.describe(percentiles=[.5, .9, .95, .99]).to_dict(),\n                                    truncated=int(values.gt(max_length).sum()))\n    (args.output / "token_lengths.json").write_text(json.dumps(\n        dict(max_length=max_length, splits=distributions), indent=2), encoding="utf-8")\n    for split, values in lengths.items():\n        plt.hist(values, bins=35, histtype="step", density=True, label=split)\n    plt.axvline(max_length, linestyle="--", label="Train-derived cutoff")\n    plt.xlabel("KoELECTRA input tokens")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(args.output / "token_lengths.png")\n    plt.close()\n\n    def encode(batch):\n        return tokenizer(batch["context"], batch["response"], truncation=True, max_length=max_length)\n\n    encoded = {}\n    for split, subset in subsets.items():\n        dataset = Dataset.from_pandas(subset[["context", "response", "labels"]], preserve_index=False)\n        encoded[split] = dataset.map(encode, batched=True, remove_columns=["context", "response"])\n    model = AutoModelForSequenceClassification.from_pretrained(\n        checkpoint, num_labels=2, label2id=LABEL_TO_ID,\n        id2label={value: name for name, value in LABEL_TO_ID.items()})\n    training = TrainingArguments(\n        output_dir=scratch, num_train_epochs=3, save_safetensors=True,\n        per_device_train_batch_size=args.batch_size, per_device_eval_batch_size=args.batch_size,\n        gradient_accumulation_steps=16 // args.batch_size,\n        learning_rate=2e-5, weight_decay=.01, warmup_ratio=.1, fp16=True, logging_steps=50,\n        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,\n        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,\n        seed=42, report_to="none",\n    )\n\n    def compute(pred):\n        return {k: v for k, v in metrics(pred.label_ids, pred.predictions.argmax(axis=-1)).items()\n                if k != "confusion"}\n\n    trainer = Trainer(model=model, args=training, train_dataset=encoded["train"],\n                      eval_dataset=encoded["validation"], processing_class=tokenizer,\n                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute)\n    trainer.train()\n    trainer.save_model(str(args.output / "final"))\n    tokenizer.save_pretrained(str(args.output / "final"))\n    pred = trainer.predict(encoded["test"])\n    predicted = pred.predictions.argmax(axis=-1)\n    errors = subsets["test"][["source_row_zero_based", "label"]].copy()\n    errors["prediction"] = [model.config.id2label[int(label)] for label in predicted]\n    errors["sarcasm_score"] = torch.as_tensor(pred.predictions).float().softmax(dim=-1)[:, 1].numpy()\n    errors["error_type"] = [("FP" if guess == 1 else "FN") if guess != actual else ""\n                            for guess, actual in zip(predicted, pred.label_ids, strict=True)]\n    errors.loc[predicted != pred.label_ids].to_csv(args.output / "errors_test.csv", index=False)\n    report = dict(test=metrics(pred.label_ids, predicted), log=trainer.state.log_history,\n                  checkpoint=checkpoint, max_length=max_length, best_validation_macro_f1=trainer.state.best_metric,\n                  best_checkpoint=Path(trainer.state.best_model_checkpoint).name,\n                  input_sha256=hashlib.sha256(args.data.read_bytes()).hexdigest(),\n                  label_to_id=LABEL_TO_ID, gpu=torch.cuda.get_device_name(),\n                  batch_size=args.batch_size, effective_batch_size=16, seed=42,\n                  package_versions={name: version(name) for name in\n                                    ("torch", "transformers", "datasets", "accelerate")})\n    (args.output / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    from sklearn.metrics import ConfusionMatrixDisplay\n    ConfusionMatrixDisplay.from_predictions(pred.label_ids, predicted, labels=[0, 1],\n                                           display_labels=list(LABEL_TO_ID))\n    plt.tight_layout()\n    plt.savefig(args.output / "confusion.png")\n    plt.close()\n    evaluations = [row for row in trainer.state.log_history if "eval_macro_f1" in row]\n    plt.plot([row["epoch"] for row in evaluations], [row["eval_macro_f1"] for row in evaluations], "o-")\n    plt.xlabel("Epoch")\n    plt.ylabel("Validation Macro F1")\n    plt.tight_layout()\n    plt.savefig(args.output / "validation.png")\n    plt.close()\n    for field in ("loss", "eval_loss"):\n        points = [(row["step"], row[field]) for row in trainer.state.log_history if field in row]\n        if points:\n            steps, values = zip(*points)\n            plt.plot(steps, values, label=field)\n    plt.xlabel("Optimizer step")\n    plt.ylabel("Loss")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(args.output / "loss.png")\n    plt.close()\n    print(json.dumps(report["test"], indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': 'import json\nfrom pathlib import Path\n\nimport joblib\n\nfrom common import format_input, normalize\n\n\ndef load_predictor(backend, path):\n    path = Path(path)\n    if backend == "baseline":\n        model = joblib.load(path)\n\n        def scores(contexts, responses):\n            inputs = [format_input(c, r) for c, r in zip(contexts, responses, strict=True)]\n            return model.predict_proba(inputs)[:, list(model.classes_).index(1)]\n\n        return scores\n    if backend != "koelectra":\n        raise ValueError("Model must be baseline or koelectra.")\n    import numpy as np\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)\n    model = AutoModelForSequenceClassification.from_pretrained(path, local_files_only=True).eval()\n    max_length = json.loads((path.parent / "token_lengths.json").read_text(encoding="utf-8"))["max_length"]\n\n    def scores(contexts, responses):\n        pairs = list(zip(contexts, responses, strict=True))\n        chunks = []\n        with torch.inference_mode():\n            for start in range(0, len(pairs), 16):\n                c, r = zip(*pairs[start:start + 16])\n                batch = tokenizer(list(map(normalize, c)), list(map(normalize, r)), padding=True,\n                                  truncation=True, max_length=max_length, return_tensors="pt")\n                chunks.append(model(**batch).logits.softmax(dim=-1)[:, 1].numpy())\n        return np.concatenate(chunks) if chunks else np.empty(0)\n\n    return scores\n', 'colab_pipeline.py': '"""Dataset preparation and held-out evaluation for the integrated notebook."""\nimport json\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.metrics import ConfusionMatrixDisplay\n\nfrom common import LABEL_TO_ID, SPLITS, normalize\nfrom train import metrics\n\n\ndef download(root):\n    from datasets import load_dataset\n    from huggingface_hub import snapshot_download\n\n    source = Path(snapshot_download("YuminKim/KoCoSa", repo_type="dataset",\n                                   allow_patterns=["data/*.parquet"], cache_dir=root / "hub")) / "data"\n    for split in SPLITS:\n        if not (source / f"{split}-00000-of-00001.parquet").is_file():\n            raise FileNotFoundError(f"KoCoSa split missing: {split}")\n    external = load_dataset("SpellOnYou/kor_sarcasm", cache_dir=str(root / "datasets"))\n    if {s: len(d) for s, d in external.items()} != {"train": 9000, "test": 301}:\n        raise ValueError("Unexpected published Kocasm split sizes")\n    for dataset in external.values():\n        if getattr(dataset.features["label"], "names", None) != ["no_sarcasm", "sarcasm"]:\n            raise ValueError("Unexpected Kocasm ClassLabel metadata; verify label semantics before evaluation")\n    expected = {"train": 10466, "validation": 1321, "test": 1037}\n    import pyarrow.parquet as parquet\n    counts = {s: parquet.ParquetFile(source / f"{s}-00000-of-00001.parquet").metadata.num_rows\n              for s in SPLITS}\n    if counts != expected:\n        raise ValueError(f"Unexpected KoCoSa split sizes: {counts}")\n    return source, {s: d.to_pandas() for s, d in external.items()}\n\n\ndef prepare_external(splits, out):\n    audit, frames = {}, []\n    for split, raw in splits.items():\n        text = raw.tokens.map(lambda value: normalize(value) if isinstance(value, str) else "")\n        frame = pd.DataFrame(dict(response=text, label=raw.label, split=split,\n                                  source_row_zero_based=range(len(raw))))\n        valid = frame[frame.response.ne("") & frame.label.isin(LABEL_TO_ID.values())]\n        groups = valid.groupby("response").label\n        audit[split] = dict(rows=len(raw), null=raw.isna().sum().to_dict(),\n                            empty_strings=int(raw.tokens.eq("").sum()),\n                            whitespace_only=int((raw.tokens.notna() & raw.tokens.ne("") & text.eq("")).sum()),\n                            exact_extra=int(raw.duplicated().sum()),\n                            normalized_extra=int(valid.response.duplicated().sum()),\n                            conflicts=int(groups.nunique().gt(1).sum()),\n                            classes=raw.label.value_counts().to_dict(),\n                            chars=text.str.len().describe().to_dict())\n        frames.append(valid)\n    data = pd.concat(frames, ignore_index=True)\n    shared = set(data.loc[data.split.eq("train"), "response"]) & set(data.loc[data.split.eq("test"), "response"])\n    conflicts = data.groupby("response").label.nunique()\n    conflicts = conflicts[conflicts.gt(1)].index\n    clean = (data[~data.response.isin(conflicts)].sort_values(["split", "source_row_zero_based"])\n             .drop_duplicates("response"))\n    audit["between_splits"] = dict(shared_inputs=len(shared), conflicting_inputs=len(conflicts),\n                                   rule="Drop conflicting labels; for consistent duplicates keep test first.")\n    for split in splits:\n        audit[split]["after_classes"] = clean.loc[clean.split.eq(split), "label"].value_counts().to_dict()\n    (out / "kocasm_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")\n    pd.DataFrame({f"{split}/{stage}": audit[split][key] for split in splits\n                  for stage, key in (("before", "classes"), ("after", "after_classes"))}).T.to_csv(\n                      out / "kocasm_preprocessing_counts.csv")\n    ax = pd.read_csv(out / "kocasm_preprocessing_counts.csv", index_col=0).plot.bar(rot=25)\n    ax.set_ylabel("Rows")\n    ax.figure.tight_layout()\n    ax.figure.savefig(out / "kocasm_classes.png")\n    plt.close(ax.figure)\n    return clean\n\n\ndef compare(kocosa, kocasm, out):\n    from inference import load_predictor\n    from transformers import AutoTokenizer\n\n    baseline = json.loads((out / "baseline/metrics.json").read_text())["context"]\n    electra = json.loads((out / "koelectra/metrics.json").read_text())\n    candidates = {\n        "baseline": (out / "baseline/baseline.joblib", baseline["test"], baseline["selection"]["validation_macro_f1"]),\n        "koelectra": (out / "koelectra/final", electra["test"], electra["best_validation_macro_f1"]),\n    }\n    winner = max(candidates, key=lambda key: candidates[key][2])\n    selection = {"model": winner, "rule": "Highest KoCoSa validation Macro F1; ties favor baseline",\n                 "scores": {name: value[2] for name, value in candidates.items()}}\n    (out / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")\n    # Exclude exact normalized text already seen by training or validation.\n    seen = kocosa.loc[kocosa.split.isin(["train", "validation"])]\n    seen_text = set(seen.response) | set(seen.context) | set(seen.context + " " + seen.response)\n    external = kocasm[kocasm.split.eq("test") & ~kocasm.response.isin(seen_text)].copy()\n    if external.label.nunique() != 2:\n        raise ValueError("External evaluation must contain both classes")\n    tokenizer = AutoTokenizer.from_pretrained(out / "koelectra/final", local_files_only=True)\n    stats = {}\n    for split, frame in kocasm.groupby("split"):\n        inputs = tokenizer([""] * len(frame), frame.response.tolist(), truncation=False, verbose=False)\n        lengths = pd.Series([len(ids) for ids in inputs["input_ids"]])\n        stats[split] = lengths.describe(percentiles=[.5, .9, .95, .99]).to_dict()\n        stats[split]["above_kocosa_cutoff"] = int(lengths.gt(electra["max_length"]).sum())\n        plt.hist(lengths, bins=30, histtype="step", density=True, label=split)\n    plt.xlabel("KoELECTRA input tokens")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(out / "kocasm_token_lengths.png")\n    plt.close()\n    (out / "kocasm_token_lengths.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")\n    rows, analysis = [], {}\n    for name, (model_path, internal, _) in candidates.items():\n        predict = load_predictor(name, model_path)\n        scores = predict([""] * len(external), external.response.tolist())\n        predicted = (scores > .5).astype(int)\n        score = metrics(external.label, predicted)\n        for dataset, values in (("KoCoSa Test", internal), ("Kocasm Test", score)):\n            rows.append(dict(model=name, dataset=dataset, **{k: v for k, v in values.items() if k != "confusion"}))\n        errors = external[["source_row_zero_based", "label"]].assign(prediction=predicted, sarcasm_score=scores)\n        errors = errors[errors.label.ne(errors.prediction)].copy()\n        errors["error_type"] = errors.prediction.map({0: "FN", 1: "FP"})\n        errors["chars"] = external.loc[errors.index, "response"].str.len()\n        errors["has_user_marker"] = external.loc[errors.index, "response"].str.contains("@user", regex=False)\n        errors.to_csv(out / f"kocasm_errors_{name}.csv", index=False)\n        analysis[name] = dict(external_rows=len(external), external_raw_rows=301,\n                              external_removed_for_kocosa_overlap=int(\n                                  kocasm[kocasm.split.eq("test")].response.isin(seen_text).sum()),\n                              macro_f1_delta=score["macro_f1"] - internal["macro_f1"],\n                              error_groups=errors.groupby("error_type").agg(\n                                  rows=("chars", "size"), mean_chars=("chars", "mean"),\n                                  user_marker_rate=("has_user_marker", "mean")).to_dict("index"))\n        ConfusionMatrixDisplay.from_predictions(external.label, predicted, labels=[0, 1])\n        plt.tight_layout()\n        plt.savefig(out / f"kocasm_confusion_{name}.png")\n        plt.close()\n    pd.DataFrame(rows).to_csv(out / "model_comparison.csv", index=False)\n    (out / "domain_analysis.json").write_text(json.dumps(analysis, indent=2), encoding="utf-8")\n    return selection\n', 'requirements-colab.txt': "pandas==2.2.3\npyarrow>=19,<24\nscikit-learn==1.7.2\nmatplotlib\njoblib\nFlask\n# Use Colab's CUDA-enabled PyTorch (2.6 or newer).\ntransformers==4.51.3\ndatasets==3.5.0\naccelerate==1.6.0\n"}.items():
    (root / name).write_text(source, encoding='utf-8')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
def run(*args):
    subprocess.run([sys.executable, '-u', *map(str, args)], cwd=root, check=True)
subprocess.run(['nvidia-smi'], check=True)
run('-m', 'pip', 'install', '-r', root / 'requirements-colab.txt')
if any(module in sys.modules for module in ('pandas', 'pyarrow', 'transformers', 'datasets')):
    print('의존성이 이미 로드됐습니다. 런타임 세션을 다시 시작하고 이 셀부터 실행하세요.')

## 2. 데이터 다운로드

KoCoSa 공식 Parquet와 Kocasm을 Colab 임시 저장소에 받습니다.

In [ ]:
from colab_pipeline import download
raw_kocosa, raw_kocasm = download(root / 'cache')
results = root / 'results' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
results.mkdir(parents=True)

## 3. 데이터 구조와 품질 검사

공식 분할, 열, 라벨 분포와 필수 필드 상태를 먼저 확인합니다.

In [ ]:
import pandas as pd
from IPython.display import display

for path in sorted(raw_kocosa.glob('*.parquet')):
    raw = pd.read_parquet(path)
    print(path.name, raw.shape, raw.label.value_counts(dropna=False).to_dict(), raw.isna().sum().to_dict())
for split, raw in raw_kocasm.items():
    print('Kocasm', split, raw.shape, raw.label.value_counts(dropna=False).to_dict(), raw.isna().sum().to_dict())

## 4. 전처리

필수값 검증, NFC·공백 정규화, 중복·충돌 처리를 적용합니다. 문장부호와 말투를 보존합니다.

In [ ]:
from colab_pipeline import prepare_external
prepared = results.parent.parent / 'prepared' / results.name
prepared.parent.mkdir(exist_ok=True)
run(root / 'preprocess.py', raw_kocosa, prepared)
data = prepared / 'kocosa_preprocessed.parquet'
audit = prepared / 'kocosa_audit.json'
shutil.copy2(audit, results / audit.name)
kocasm = prepare_external(raw_kocasm, results)

## 5. 공식 분할 구성

Train·Validation·Test를 유지하고, 라벨을 두 모델에 동일하게 매핑합니다.

In [ ]:
from common import LABEL_TO_ID, SPLITS
kocosa = pd.read_parquet(data)
assert set(kocosa.split) == set(SPLITS)
assert set(kocosa.label) == set(LABEL_TO_ID)
assert not kocosa.duplicated(['context', 'response']).any()
assert 'sarcasm_explanation' not in kocosa
display(pd.crosstab(kocosa.split, kocosa.label))

## 6. 베이스라인 학습

기존 TF-IDF와 로지스틱 회귀를 같은 설정으로 다시 학습합니다. 각 결과는 새 실험 폴더에 저장됩니다.

In [ ]:
run(root / 'train.py', data, audit, results / 'baseline')

## 7. KoELECTRA 파인튜닝

Train 토큰 길이로 입력 길이를 정하고 Validation Macro F1으로 최적 체크포인트를 고릅니다. 메모리 부족 시 새 실험에서 배치를 4 또는 2로 낮춥니다.

In [ ]:
run(root / 'finetune.py', data, '--output', results / 'koelectra', '--batch-size', '8')

## 8. KoCoSa 평가 비교

같은 KoCoSa Test에서 나온 실제 지표를 비교합니다.

In [ ]:
baseline = json.loads((results / 'baseline/metrics.json').read_text())['context']
koelectra = json.loads((results / 'koelectra/metrics.json').read_text())
display(pd.DataFrame({'baseline': baseline['test'], 'koelectra': koelectra['test']}).drop(index='confusion'))

## 9. Kocasm 외부 평가

두 모델에 빈 맥락과 동일한 단일 문장을 제공합니다. Kocasm Train은 품질 감사 전용이며 외부 결과는 모델 선택에 사용하지 않습니다.

In [ ]:
from colab_pipeline import compare
selection = compare(kocosa, kocasm, results)
display(pd.read_csv(results / 'model_comparison.csv'))

## 10. 최종 모델 저장

두 모델의 KoCoSa Validation Macro F1으로 선택합니다. 이미 저장한 가중치를 가리켜 중복 복사를 피합니다.

In [ ]:
artifacts = {'baseline': 'baseline/baseline.joblib', 'koelectra': 'koelectra/final'}
selected = results / artifacts[selection['model']]
assert selected.exists()
(results / 'selected_model.txt').write_text(artifacts[selection['model']], encoding='utf-8')
print(selection, selected)

## 11. 저장한 모델로 추론

두 모델을 디스크에서 다시 로드하여 동일한 입력으로 확인합니다.

In [ ]:
from inference import load_predictor
for name, relative in artifacts.items():
    predictor = load_predictor(name, results / relative)
    score = float(predictor(['A: 오늘도 한 시간 늦었네.'], ['B: 정말 시간 약속 잘 지킨다.'])[0])
    assert 0 <= score <= 1
    print(name, '비꼼' if score > .5 else '비꼼 아님', round(score, 4))

## 12. 결과와 그래프 보관

기본값은 Colab 임시 저장입니다. 저장을 선택하면 최종 KoELECTRA와 실험 산출물을 개인 Drive에 한 벌 복사합니다.

In [ ]:
from IPython.display import Image
for name in ('baseline/preprocessing.png', 'koelectra/loss.png', 'koelectra/token_lengths.png'):
    display(Image(filename=str(results / name)))
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    home = Path('/content/drive/MyDrive')
    size = sum(path.stat().st_size for path in results.rglob('*') if path.is_file())
    print(f'복사 예정 용량: {size / 1_000_000:.1f} MB')
    if size > min(4_000_000_000, shutil.disk_usage(home).free):
        raise RuntimeError('Google Drive 여유 공간이 부족합니다.')
    destination = home / 'NoonchiCoach' / results.name
    destination.parent.mkdir(exist_ok=True)
    shutil.copytree(results, destination)
    print(destination)
else:
    print('이 폴더를 세션 종료 전에 내려받을 수 있습니다:', results)

## 13. 저장된 KoELECTRA 임계값 분석

기존 실험이 완료됐으면 **이 단계의 코드 셀만** 실행합니다. 새 세션에서는 아래 두 경로를 실제 경로로 지정합니다. 실험 폴더는 `koelectra/final`과 `koelectra/metrics.json`을 포함해야 하며, Parquet는 해당 실행에 사용한 전처리 파일이어야 합니다. Drive 경로를 쓰는 경우 먼저 Drive를 연결합니다.

Validation에서 0.05–0.95를 0.01 간격으로 검사합니다. 기본 0.50 대비 Macro F1 하락을 최대 0.01로 제한하면서 FPR이 가장 낮은 값을 선택합니다. 동률이면 Macro F1, Recall, 낮은 임계값 순입니다. 두 Test는 선택 완료 후 평가합니다. 모델과 기존 출력은 읽기 전용으로 사용하고, 분석은 별도의 타임스탬프 폴더에 저장합니다. FP/FN CSV는 원문 대신 원본 행 번호와 오류 특성을 기록합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

EXISTING_RUN = globals().get('results')  # 새 세션 예: Path('/content/drive/MyDrive/NoonchiCoach/실험ID')
PREPARED_PARQUET = globals().get('data')  # 새 세션: 해당 실험의 전처리 Parquet 경로
ANALYSIS_ROOT = Path('/content/noonchi_threshold_results')  # 필요하면 Drive의 별도 경로 지정
if not EXISTING_RUN or not PREPARED_PARQUET:
    raise ValueError('EXISTING_RUN과 PREPARED_PARQUET에 실제 경로를 지정하세요.')
if not (Path(EXISTING_RUN) / 'koelectra/final').is_dir() or not Path(PREPARED_PARQUET).is_file():
    raise FileNotFoundError('기존 가중치 폴더와 전처리 Parquet 경로를 확인하세요.')
code_dir = Path('/content/noonchi_threshold_code')
code_dir.mkdir(exist_ok=True)
for name, source in {'common.py': 'import unicodedata\n\nLABEL_TO_ID = {"Non-Sarcasm": 0, "Sarcasm": 1}\nSPLITS = ("train", "validation", "test")\n\n\ndef normalize(value):\n    return " ".join(unicodedata.normalize("NFC", value).split())\n\n\ndef format_input(context, response):\n    return f"맥락: {normalize(context)}\\n마지막 발언: {normalize(response)}"\n', 'train.py': 'import json\nfrom pathlib import Path\nimport sys\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score\nfrom sklearn.pipeline import make_pipeline\n\nfrom common import LABEL_TO_ID, SPLITS, format_input\n\n\ndef metrics(y, pred):\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return dict(accuracy=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average="macro"),\n                precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),\n                fpr=float(fp / (tn + fp)), confusion=[[int(tn), int(fp)], [int(fn), int(tp)]])\n\n\ndef run(source, audit_file=None, output="results"):\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=True)\n    frame = pd.read_parquet(source)\n    frame["target"] = frame.label.map(LABEL_TO_ID)\n    assert frame.target.notna().all() and not frame.duplicated(["context", "response"]).any()\n    train, val, test = [frame[frame.split.eq(s)].reset_index(drop=True) for s in SPLITS]\n    results = {}\n    for mode in ("context", "response_only"):\n        def features(rows):\n            return [format_input(c if mode == "context" else "", r) for c, r in zip(rows.context, rows.response)]\n\n        x_train, x_val, x_test = map(features, (train, val, test))\n        trials = []\n        best = None\n        for c in (.25, 1, 4):\n            for weight in (None, "balanced"):\n                model = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2,\n                                                       max_features=85000, sublinear_tf=True),\n                                      LogisticRegression(C=c, class_weight=weight, max_iter=500, random_state=42))\n                model.fit(x_train, train.target)\n                score = f1_score(val.target, model.predict(x_val), average="macro")\n                trials.append(dict(C=c, class_weight=weight, validation_macro_f1=score))\n                if best is None or score > best[0]:\n                    best = (score, model, trials[-1])\n        _, model, selection = best\n        pred = model.predict(x_test)\n        results[mode] = dict(selection=selection, trials=trials, test=metrics(test.target, pred))\n        errors = test.loc[test.target.ne(pred), ["source_row_zero_based", "label"]].copy()\n        errors["prediction"] = pred[test.target.ne(pred)]\n        errors["score"] = model.predict_proba(x_test)[test.target.ne(pred), 1]\n        errors.to_csv(out / f"errors_{mode}.csv", index=False)\n        fig, ax = plt.subplots()\n        ConfusionMatrixDisplay(confusion_matrix(test.target, pred), display_labels=["Non-Sarcasm", "Sarcasm"]).plot(ax=ax)\n        fig.tight_layout()\n        fig.savefig(out / f"confusion_{mode}.png", dpi=135)\n        plt.close(fig)\n        if mode == "context":\n            joblib.dump(model, out / "baseline.joblib", compress=3)\n    fig, ax = plt.subplots()\n    for mode, result in results.items():\n        for weight in (None, "balanced"):\n            trials = [x for x in result["trials"] if x["class_weight"] == weight]\n            ax.plot([x["C"] for x in trials], [x["validation_macro_f1"] for x in trials],\n                    "o-", label=f"{mode} / {weight or \'unweighted\'}")\n    ax.set(xscale="log", xlabel="Logistic regression C", ylabel="Validation Macro F1")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / "validation.png", dpi=135)\n    plt.close(fig)\n    if audit_file:\n        audit = json.loads(Path(audit_file).read_text(encoding="utf-8"))\n        after = pd.crosstab(frame.split, frame.label)\n        before = pd.DataFrame({s: data["labels"] for s, data in audit["splits"].items()}).T\n        comparison = pd.concat({"before": before, "after": after}, names=["stage", "split"])\n        comparison.to_csv(out / "preprocessing_counts.csv")\n        ax = comparison.plot.bar(rot=30)\n        ax.set_ylabel("Rows")\n        ax.figure.tight_layout()\n        ax.figure.savefig(out / "preprocessing.png", dpi=135)\n        plt.close(ax.figure)\n    (out / "metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")\n    print(json.dumps(results, indent=2))\n\n\nif __name__ == "__main__":\n    run(*sys.argv[1:])\n', 'inference.py': 'import json\nfrom pathlib import Path\n\nimport joblib\n\nfrom common import format_input, normalize\n\n\ndef load_predictor(backend, path):\n    path = Path(path)\n    if backend == "baseline":\n        model = joblib.load(path)\n\n        def scores(contexts, responses):\n            inputs = [format_input(c, r) for c, r in zip(contexts, responses, strict=True)]\n            return model.predict_proba(inputs)[:, list(model.classes_).index(1)]\n\n        return scores\n    if backend != "koelectra":\n        raise ValueError("Model must be baseline or koelectra.")\n    import numpy as np\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)\n    model = AutoModelForSequenceClassification.from_pretrained(path, local_files_only=True).eval()\n    max_length = json.loads((path.parent / "token_lengths.json").read_text(encoding="utf-8"))["max_length"]\n\n    def scores(contexts, responses):\n        pairs = list(zip(contexts, responses, strict=True))\n        chunks = []\n        with torch.inference_mode():\n            for start in range(0, len(pairs), 16):\n                c, r = zip(*pairs[start:start + 16])\n                batch = tokenizer(list(map(normalize, c)), list(map(normalize, r)), padding=True,\n                                  truncation=True, max_length=max_length, return_tensors="pt")\n                chunks.append(model(**batch).logits.softmax(dim=-1)[:, 1].numpy())\n        return np.concatenate(chunks) if chunks else np.empty(0)\n\n    return scores\n', 'colab_pipeline.py': '"""Dataset preparation and held-out evaluation for the integrated notebook."""\nimport json\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom sklearn.metrics import ConfusionMatrixDisplay\n\nfrom common import LABEL_TO_ID, SPLITS, normalize\nfrom train import metrics\n\n\ndef download(root):\n    from datasets import load_dataset\n    from huggingface_hub import snapshot_download\n\n    source = Path(snapshot_download("YuminKim/KoCoSa", repo_type="dataset",\n                                   allow_patterns=["data/*.parquet"], cache_dir=root / "hub")) / "data"\n    for split in SPLITS:\n        if not (source / f"{split}-00000-of-00001.parquet").is_file():\n            raise FileNotFoundError(f"KoCoSa split missing: {split}")\n    external = load_dataset("SpellOnYou/kor_sarcasm", cache_dir=str(root / "datasets"))\n    if {s: len(d) for s, d in external.items()} != {"train": 9000, "test": 301}:\n        raise ValueError("Unexpected published Kocasm split sizes")\n    for dataset in external.values():\n        if getattr(dataset.features["label"], "names", None) != ["no_sarcasm", "sarcasm"]:\n            raise ValueError("Unexpected Kocasm ClassLabel metadata; verify label semantics before evaluation")\n    expected = {"train": 10466, "validation": 1321, "test": 1037}\n    import pyarrow.parquet as parquet\n    counts = {s: parquet.ParquetFile(source / f"{s}-00000-of-00001.parquet").metadata.num_rows\n              for s in SPLITS}\n    if counts != expected:\n        raise ValueError(f"Unexpected KoCoSa split sizes: {counts}")\n    return source, {s: d.to_pandas() for s, d in external.items()}\n\n\ndef prepare_external(splits, out):\n    audit, frames = {}, []\n    for split, raw in splits.items():\n        text = raw.tokens.map(lambda value: normalize(value) if isinstance(value, str) else "")\n        frame = pd.DataFrame(dict(response=text, label=raw.label, split=split,\n                                  source_row_zero_based=range(len(raw))))\n        valid = frame[frame.response.ne("") & frame.label.isin(LABEL_TO_ID.values())]\n        groups = valid.groupby("response").label\n        audit[split] = dict(rows=len(raw), null=raw.isna().sum().to_dict(),\n                            empty_strings=int(raw.tokens.eq("").sum()),\n                            whitespace_only=int((raw.tokens.notna() & raw.tokens.ne("") & text.eq("")).sum()),\n                            exact_extra=int(raw.duplicated().sum()),\n                            normalized_extra=int(valid.response.duplicated().sum()),\n                            conflicts=int(groups.nunique().gt(1).sum()),\n                            classes=raw.label.value_counts().to_dict(),\n                            chars=text.str.len().describe().to_dict())\n        frames.append(valid)\n    data = pd.concat(frames, ignore_index=True)\n    shared = set(data.loc[data.split.eq("train"), "response"]) & set(data.loc[data.split.eq("test"), "response"])\n    conflicts = data.groupby("response").label.nunique()\n    conflicts = conflicts[conflicts.gt(1)].index\n    clean = (data[~data.response.isin(conflicts)].sort_values(["split", "source_row_zero_based"])\n             .drop_duplicates("response"))\n    audit["between_splits"] = dict(shared_inputs=len(shared), conflicting_inputs=len(conflicts),\n                                   rule="Drop conflicting labels; for consistent duplicates keep test first.")\n    for split in splits:\n        audit[split]["after_classes"] = clean.loc[clean.split.eq(split), "label"].value_counts().to_dict()\n    (out / "kocasm_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")\n    pd.DataFrame({f"{split}/{stage}": audit[split][key] for split in splits\n                  for stage, key in (("before", "classes"), ("after", "after_classes"))}).T.to_csv(\n                      out / "kocasm_preprocessing_counts.csv")\n    ax = pd.read_csv(out / "kocasm_preprocessing_counts.csv", index_col=0).plot.bar(rot=25)\n    ax.set_ylabel("Rows")\n    ax.figure.tight_layout()\n    ax.figure.savefig(out / "kocasm_classes.png")\n    plt.close(ax.figure)\n    return clean\n\n\ndef compare(kocosa, kocasm, out):\n    from inference import load_predictor\n    from transformers import AutoTokenizer\n\n    baseline = json.loads((out / "baseline/metrics.json").read_text())["context"]\n    electra = json.loads((out / "koelectra/metrics.json").read_text())\n    candidates = {\n        "baseline": (out / "baseline/baseline.joblib", baseline["test"], baseline["selection"]["validation_macro_f1"]),\n        "koelectra": (out / "koelectra/final", electra["test"], electra["best_validation_macro_f1"]),\n    }\n    winner = max(candidates, key=lambda key: candidates[key][2])\n    selection = {"model": winner, "rule": "Highest KoCoSa validation Macro F1; ties favor baseline",\n                 "scores": {name: value[2] for name, value in candidates.items()}}\n    (out / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")\n    # Exclude exact normalized text already seen by training or validation.\n    seen = kocosa.loc[kocosa.split.isin(["train", "validation"])]\n    seen_text = set(seen.response) | set(seen.context) | set(seen.context + " " + seen.response)\n    external = kocasm[kocasm.split.eq("test") & ~kocasm.response.isin(seen_text)].copy()\n    if external.label.nunique() != 2:\n        raise ValueError("External evaluation must contain both classes")\n    tokenizer = AutoTokenizer.from_pretrained(out / "koelectra/final", local_files_only=True)\n    stats = {}\n    for split, frame in kocasm.groupby("split"):\n        inputs = tokenizer([""] * len(frame), frame.response.tolist(), truncation=False, verbose=False)\n        lengths = pd.Series([len(ids) for ids in inputs["input_ids"]])\n        stats[split] = lengths.describe(percentiles=[.5, .9, .95, .99]).to_dict()\n        stats[split]["above_kocosa_cutoff"] = int(lengths.gt(electra["max_length"]).sum())\n        plt.hist(lengths, bins=30, histtype="step", density=True, label=split)\n    plt.xlabel("KoELECTRA input tokens")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(out / "kocasm_token_lengths.png")\n    plt.close()\n    (out / "kocasm_token_lengths.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")\n    rows, analysis = [], {}\n    for name, (model_path, internal, _) in candidates.items():\n        predict = load_predictor(name, model_path)\n        scores = predict([""] * len(external), external.response.tolist())\n        predicted = (scores > .5).astype(int)\n        score = metrics(external.label, predicted)\n        for dataset, values in (("KoCoSa Test", internal), ("Kocasm Test", score)):\n            rows.append(dict(model=name, dataset=dataset, **{k: v for k, v in values.items() if k != "confusion"}))\n        errors = external[["source_row_zero_based", "label"]].assign(prediction=predicted, sarcasm_score=scores)\n        errors = errors[errors.label.ne(errors.prediction)].copy()\n        errors["error_type"] = errors.prediction.map({0: "FN", 1: "FP"})\n        errors["chars"] = external.loc[errors.index, "response"].str.len()\n        errors["has_user_marker"] = external.loc[errors.index, "response"].str.contains("@user", regex=False)\n        errors.to_csv(out / f"kocasm_errors_{name}.csv", index=False)\n        analysis[name] = dict(external_rows=len(external), external_raw_rows=301,\n                              external_removed_for_kocosa_overlap=int(\n                                  kocasm[kocasm.split.eq("test")].response.isin(seen_text).sum()),\n                              macro_f1_delta=score["macro_f1"] - internal["macro_f1"],\n                              error_groups=errors.groupby("error_type").agg(\n                                  rows=("chars", "size"), mean_chars=("chars", "mean"),\n                                  user_marker_rate=("has_user_marker", "mean")).to_dict("index"))\n        ConfusionMatrixDisplay.from_predictions(external.label, predicted, labels=[0, 1])\n        plt.tight_layout()\n        plt.savefig(out / f"kocasm_confusion_{name}.png")\n        plt.close()\n    pd.DataFrame(rows).to_csv(out / "model_comparison.csv", index=False)\n    (out / "domain_analysis.json").write_text(json.dumps(analysis, indent=2), encoding="utf-8")\n    return selection\n', 'threshold_analysis.py': '"""Post-training threshold analysis. Never fits or modifies a model."""\nimport json\nimport hashlib\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import ConfusionMatrixDisplay\n\nfrom common import LABEL_TO_ID\nfrom inference import load_predictor\nfrom train import metrics\nfrom colab_pipeline import download, prepare_external\n\n\ndef select_threshold(labels, scores, out):\n    sweep = pd.DataFrame([\n        dict(threshold=value, **metrics(labels, scores > value))\n        for value in np.arange(5, 96, 1) / 100\n    ])\n    reference = sweep.loc[sweep.threshold.eq(.5)].iloc[0]\n    eligible = sweep[sweep.macro_f1.ge(reference.macro_f1 - .01 - 1e-12)\n                     & sweep.fpr.le(reference.fpr)]\n    selected = eligible.sort_values(\n        ["fpr", "macro_f1", "recall", "threshold"], ascending=[True, False, False, True]\n    ).iloc[0]\n    sweep.drop(columns="confusion").to_csv(out / "validation_thresholds.csv", index=False)\n    ax = sweep.set_index("threshold")[["precision", "recall", "fpr", "macro_f1"]].plot()\n    ax.axvline(selected.threshold, color="black", linestyle="--", label="Selected on validation")\n    ax.set_ylim(0, 1.02)\n    ax.legend()\n    ax.figure.tight_layout()\n    ax.figure.savefig(out / "validation_thresholds.png")\n    plt.close(ax.figure)\n    return float(selected.threshold)\n\n\ndef evaluate_saved(run, prepared, cache, out):\n    run, prepared, out = map(Path, (run, prepared, out))\n    metadata = json.loads((run / "koelectra/metrics.json").read_text(encoding="utf-8"))\n    if hashlib.sha256(prepared.read_bytes()).hexdigest() != metadata["input_sha256"]:\n        raise ValueError("Use the exact preprocessed Parquet from this training run.")\n    classifier = load_predictor("koelectra", run / "koelectra/final")\n    frame = pd.read_parquet(prepared)\n    frame["target"] = frame.label.map(LABEL_TO_ID)\n    out.mkdir(parents=True, exist_ok=False)\n    validation = frame[frame.split.eq("validation")]\n    if set(validation.target) != {0, 1}:\n        raise ValueError("KoCoSa Validation must contain both mapped labels.")\n    scores = classifier(validation.context.tolist(), validation.response.tolist())\n    threshold = select_threshold(validation.target, scores, out)\n    validation[["source_row_zero_based", "target"]].assign(score=scores).to_csv(\n        out / "validation_scores.csv", index=False)\n    (out / "selection.json").write_text(json.dumps({\n        "threshold": threshold, "reference_threshold": .5,\n        "rule": "Minimize validation FPR with Macro F1 within 0.01 of the 0.50 reference and FPR no higher. Ties favor F1, recall, then lower threshold.",\n        "model_path": str(run / "koelectra/final"),\n    }, indent=2), encoding="utf-8")\n\n    _, raw_external = download(Path(cache))\n    clean = prepare_external(raw_external, out)\n    seen = frame[frame.split.isin(["train", "validation"])]\n    seen_text = set(seen.response) | set(seen.context) | set(seen.context + " " + seen.response)\n    external = clean[clean.split.eq("test") & ~clean.response.isin(seen_text)].copy()\n    external["context"], external["target"] = "", external.label\n    (out / "kocasm_input_check.json").write_text(json.dumps({\n        "label_mapping": {"0": "no_sarcasm", "1": "sarcasm"},\n        "metadata_verified_by_download": True, "empty_context": bool(external.context.eq("").all()),\n        "raw_test_rows": len(raw_external["test"]), "evaluated_test_rows": len(external),\n        "class_counts": external.target.value_counts().to_dict(),\n    }, indent=2), encoding="utf-8")\n    rows = []\n    for name, test in (("kocosa", frame[frame.split.eq("test")]), ("kocasm", external)):\n        if set(test.target) != {0, 1}:\n            raise ValueError(f"{name} Test must contain both mapped labels.")\n        scores = classifier(test.context.tolist(), test.response.tolist())\n        for tag, cutoff in (("default", .5), ("selected", threshold)):\n            predicted = (scores > cutoff).astype(int)\n            values = metrics(test.target, predicted)\n            rows.append(dict(dataset=name, policy=tag, threshold=cutoff, n=len(test),\n                             predicted_positive=int(predicted.sum()),\n                             **{k: v for k, v in values.items() if k != "confusion"}))\n            errors = test[["source_row_zero_based", "target"]].assign(\n                score=scores, prediction=predicted, chars=test.response.str.len(),\n                has_user_marker=test.response.str.contains("@user", regex=False))\n            errors = errors[errors.target.ne(errors.prediction)].copy()\n            errors["error_type"] = errors.prediction.map({0: "FN", 1: "FP"})\n            errors["distance_to_threshold"] = (errors.score - cutoff).abs()\n            errors.sort_values("distance_to_threshold").to_csv(out / f"{name}_{tag}_errors.csv", index=False)\n            errors.groupby("error_type").agg(count=("score", "size"), mean_score=("score", "mean"),\n                mean_chars=("chars", "mean"), user_marker_rate=("has_user_marker", "mean")).to_csv(\n                    out / f"{name}_{tag}_error_summary.csv")\n            ConfusionMatrixDisplay.from_predictions(test.target, predicted, labels=[0, 1],\n                                                   display_labels=list(LABEL_TO_ID))\n            plt.tight_layout()\n            plt.savefig(out / f"{name}_{tag}_confusion.png")\n            plt.close()\n    table = pd.DataFrame(rows)\n    table.to_csv(out / "test_comparison.csv", index=False)\n    ax = table.set_index(["dataset", "policy"])[["macro_f1", "precision", "recall", "fpr"]].plot.bar(rot=20)\n    ax.set_ylim(0, 1.02)\n    ax.figure.tight_layout()\n    ax.figure.savefig(out / "test_comparison.png")\n    plt.close(ax.figure)\n    return table\n', 'requirements-colab.txt': "pandas==2.2.3\npyarrow>=19,<24\nscikit-learn==1.7.2\nmatplotlib\njoblib\nFlask\n# Use Colab's CUDA-enabled PyTorch (2.6 or newer).\ntransformers==4.51.3\ndatasets==3.5.0\naccelerate==1.6.0\n"}.items():
    (code_dir / name).write_text(source, encoding='utf-8')
if 'colab_pipeline' not in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(code_dir / 'requirements-colab.txt')], check=True)
analysis = ANALYSIS_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
subprocess.run([sys.executable, '-c',
    'import sys; from threshold_analysis import evaluate_saved; print(evaluate_saved(*sys.argv[1:]).to_string(index=False))',
    str(EXISTING_RUN), str(PREPARED_PARQUET), '/content/noonchi_complete/cache', str(analysis)],
    cwd=code_dir, check=True)
print('분석 결과:', analysis)